# Cross-Market Information Uptake

## Paper Narrative

The main story here is that **more liquid external markets may act as fast public-information channels for Polymarket**.

This benchmark answers the supporting paper question:
- is there a lead-lag signal between crypto shocks and future Polymarket repricing,
- does Polymarket sometimes absorb information more slowly than a liquid external market,
- and is the multimodal story with external covariates justified at all.

For the paper, this is a supporting benchmark rather than the final main result: it tests whether external markets contain usable directional information before we build stronger hybrid models.

## Everyday Intuition: Why This Might Work

The intuition is simple:
- `BTC/ETH` markets are more liquid and often move faster;
- for some event classes, Polymarket may update more slowly;
- if the external market has already reacted and the prediction market has not, that disagreement may foreshadow a future repricing.

## What Data We Will Use

- the repricing dataset from Polymarket;
- aligned `btc_usd` and `eth_usd` 5-minute data;
- crypto returns, rolling volatility, and shock features;
- a backward as-of join on the benchmark snapshot timestamp.

## What Metrics To Track

- descriptive repricing rate by shock buckets;
- `average_precision`, `roc_auc`, and `log_loss` for the predictive part;
- comparison of `market-only`, `crypto-only`, and `hybrid` baselines.

## What Models To Train

- a `market-only` baseline;
- a `crypto-only` baseline;
- a `hybrid` baseline;
- initially simple logistic models, just to test whether there is signal at all.

## How To Read This Notebook

This is a supporting research tutorial about the external information channel.

The logic is:
1. We take the already defined repricing task.
2. We join `BTC/ETH 5m` as a more liquid external market.
3. We first check descriptive evidence: does repricing increase after crypto shocks?
4. Then we compare `market-only`, `crypto-only`, and `hybrid` baselines.

## Task

**What we test:**
- whether a more liquid crypto market can serve as a fast external carrier of information for Polymarket.

**Input:**
- the standard repricing-benchmark snapshot;
- aligned `BTC/ETH` returns and volatility features.

**Target:**
- the same repricing target as in the main benchmark.

**Why this task matters:**
- if an external liquid market truly carries a lead signal, that supports the multimodal forecasting story;
- if not, then crypto should be used more selectively, through gating or trust logic.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "benchmarks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "benchmarks"))

from benchmark_utils import (
    build_repricing_dataset,
    connect,
    load_eligible_markets,
    load_probabilities_for_markets,
    rolling_time_splits,
)
from covariate_utils import asof_join_covariates, load_external_covariates

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)

DOMAINS = ["geopolitics", "finance_economy"]
MAX_MARKETS_PER_DOMAIN = 150
FUTURE_HORIZON_HOURS = 24
LOOKBACK_HOURS = 24
SAMPLE_EVERY_HOURS = 24
MOVE_THRESHOLD = 0.15
DB_PATH = REPO_ROOT / "db" / "resolved_probability_dataset.sqlite"
COVARIATE_PATH = REPO_ROOT / "data" / "external_covariates"


## Dataset Construction

Here we do not create a new base task from scratch, but extend the existing repricing dataset.

Steps:
- build the repricing dataset from Polymarket;
- load `btc_usd` and `eth_usd` 5-minute data;
- align them in time through a backward as-of join;
- compute returns, rolling volatility, and shock features.

**Unit of evaluation:**
- one row from the repricing dataset + external crypto context available at the same timestamp.

In [2]:
conn = connect(DB_PATH)
markets_df = pd.concat(
    [
        load_eligible_markets(conn, domain=domain, max_markets=MAX_MARKETS_PER_DOMAIN)
        for domain in DOMAINS
    ],
    ignore_index=True,
)
probabilities_df = load_probabilities_for_markets(conn, markets_df["market_id"].tolist())
repricing_df = build_repricing_dataset(
    markets_df,
    probabilities_df,
    future_horizon_hours=FUTURE_HORIZON_HOURS,
    lookback_hours=LOOKBACK_HOURS,
    sample_every_hours=SAMPLE_EVERY_HOURS,
    move_threshold=MOVE_THRESHOLD,
)

summary = pd.DataFrame(
    {
        "domains": [", ".join(DOMAINS)],
        "markets": [markets_df["market_id"].nunique()],
        "probability_rows": [len(probabilities_df)],
        "repricing_rows": [len(repricing_df)],
        "repricing_event_rate": [repricing_df["target"].mean()],
        "start": [repricing_df["timestamp_utc"].min()],
        "end": [repricing_df["timestamp_utc"].max()],
    }
)
display(summary)
repricing_df.head()


## Evaluation Protocol

We look at the task from two angles.

1. **Descriptive analysis**
- does repricing rate increase in the top BTC/ETH shock buckets;
- is there visible evidence linking crypto moves and future Polymarket moves.

2. **Predictive analysis**
- `market-only` baseline;
- `crypto-only` baseline;
- `hybrid` baseline.

The main metric is the same as in the repricing benchmark: the goal is not only to guess the class, but to understand whether the external market adds information.

In [3]:
covariates_df = load_external_covariates(COVARIATE_PATH)
covariates_df = covariates_df[covariates_df["series_id"].isin(["btc_usd", "eth_usd"])].copy()

price_wide = (
    covariates_df.pivot_table(
        index="timestamp_utc",
        columns="series_id",
        values="close",
        aggfunc="last",
    )
    .sort_index()
)
returns = price_wide.pct_change()

crypto_features = pd.DataFrame({"timestamp_utc": price_wide.index})
for series_id in ["btc_usd", "eth_usd"]:
    r = returns[series_id]
    crypto_features[f"{series_id}_ret_5m"] = r.values
    crypto_features[f"{series_id}_ret_1h"] = r.rolling(12).sum().values
    crypto_features[f"{series_id}_ret_6h"] = r.rolling(72).sum().values
    crypto_features[f"{series_id}_ret_24h"] = r.rolling(288).sum().values
    crypto_features[f"{series_id}_vol_1h"] = r.rolling(12).std().values
    crypto_features[f"{series_id}_vol_6h"] = r.rolling(72).std().values
    crypto_features[f"{series_id}_abs_ret_1h"] = r.abs().rolling(12).sum().values
    crypto_features[f"{series_id}_abs_ret_6h"] = r.abs().rolling(72).sum().values

crypto_features["btc_eth_ret_gap_1h"] = (
    crypto_features["btc_usd_ret_1h"] - crypto_features["eth_usd_ret_1h"]
)
crypto_features["btc_eth_vol_gap_6h"] = (
    crypto_features["btc_usd_vol_6h"] - crypto_features["eth_usd_vol_6h"]
)

joined_df = asof_join_covariates(
    repricing_df,
    crypto_features,
    base_time_col="timestamp_utc",
    max_age="10min",
)

crypto_cols = [col for col in joined_df.columns if col.startswith("btc_") or col.startswith("eth_")]
market_cols = [
    "current_yes_probability",
    "confidence_margin",
    "hours_to_resolution",
    "life_progress",
    "recent_abs_move_mean",
    "recent_abs_move_max",
    "recent_volatility",
    "recent_directional_move",
    "observed_trade_share",
    "trade_count_sum",
    "total_size_sum",
]
market_cols = [col for col in market_cols if col in joined_df.columns]

display(
    pd.DataFrame(
        {
            "joined_rows": [len(joined_df)],
            "joined_markets": [joined_df["market_id"].nunique()],
            "crypto_feature_count": [len(crypto_cols)],
            "market_feature_count": [len(market_cols)],
            "btc_missing_share": [joined_df["btc_usd_ret_1h"].isna().mean()],
        }
    )
)
joined_df.head()


## Results

Here we need to separate two different conclusions:
- whether there is any descriptive evidence for information uptake at all;
- whether that signal is strong enough to improve the predictive baseline.

This distinction matters: an external market can be scientifically interesting even when naive feature concatenation still does not produce a strong gain.

In [4]:
shock_source = joined_df["btc_usd_abs_ret_1h"].fillna(0.0)
shock_bins = pd.qcut(shock_source, q=5, duplicates="drop")
shock_table = (
    joined_df.assign(btc_shock_bin=shock_bins)
    .groupby("btc_shock_bin", observed=False)
    .agg(
        repricing_rate=("target", "mean"),
        avg_abs_future_move=("future_move", lambda s: np.abs(s).mean()),
        rows=("target", "size"),
    )
    .reset_index()
)
display(shock_table)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=shock_table, x="btc_shock_bin", y="repricing_rate", ax=axes[0], color="#4C72B0")
axes[0].set_title("Repricing Rate by BTC 1h Shock Quintile")
axes[0].set_xlabel("BTC absolute 1h return quintile")
axes[0].set_ylabel("Future repricing rate")
axes[0].tick_params(axis="x", rotation=30)

sns.barplot(data=shock_table, x="btc_shock_bin", y="avg_abs_future_move", ax=axes[1], color="#55A868")
axes[1].set_title("Average |Future Move| by BTC 1h Shock Quintile")
axes[1].set_xlabel("BTC absolute 1h return quintile")
axes[1].set_ylabel("Average absolute future move")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()

sample_df = joined_df.dropna(subset=["btc_usd_ret_1h", "future_move"]).sample(
    n=min(2000, len(joined_df.dropna(subset=["btc_usd_ret_1h", "future_move"]))),
    random_state=42,
)
plt.figure(figsize=(6, 4))
sns.scatterplot(data=sample_df, x="btc_usd_ret_1h", y="future_move", alpha=0.25, s=18)
plt.title("BTC 1h Return vs Polymarket Future Move")
plt.xlabel("BTC 1h return")
plt.ylabel("Future Polymarket move over 24h")
plt.tight_layout()


In [5]:
work_df = joined_df.dropna(subset=["target"]).copy()
splits = rolling_time_splits(work_df, time_col="timestamp_utc", n_splits=4, min_train_fraction=0.5)

model_specs = {
    "market_logistic": market_cols,
    "crypto_logistic": crypto_cols,
    "hybrid_logistic": market_cols + crypto_cols,
}

metric_rows = []
for model_name, feature_cols in model_specs.items():
    for train_df, test_df, meta in splits:
        pipeline = Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(max_iter=500, class_weight="balanced")),
            ]
        )
        y_train = train_df["target"].astype(int)
        y_test = test_df["target"].astype(int)
        pipeline.fit(train_df[feature_cols], y_train)
        p_test = pipeline.predict_proba(test_df[feature_cols])[:, 1]
        metric_rows.append(
            {
                "model": model_name,
                "fold": meta["fold"],
                "roc_auc": roc_auc_score(y_test, p_test),
                "average_precision": average_precision_score(y_test, p_test),
                "log_loss": log_loss(y_test, np.clip(p_test, 1e-6, 1.0 - 1e-6)),
            }
        )

metrics_df = pd.DataFrame(metric_rows)
metrics_summary = (
    metrics_df.groupby("model")[["roc_auc", "average_precision", "log_loss"]]
    .agg(["mean", "std"])
    .round(4)
)
display(metrics_summary)

plot_df = metrics_df.melt(id_vars=["model", "fold"], var_name="metric", value_name="value")
g = sns.catplot(
    data=plot_df,
    x="model",
    y="value",
    col="metric",
    kind="bar",
    sharey=False,
    height=4,
    aspect=1.1,
)
g.set_xticklabels(rotation=25)
g.fig.suptitle("Cross-Market Uptake Baselines", y=1.05)
plt.show()


## Interpretation

This notebook is best read as a hypothesis check, not as the final main result.

If `crypto-only` is nearly random, but shock buckets show elevated repricing rates, that means:
- the signal exists,
- but it does not work as a standalone predictor,
- and it is better used as a **conditional update signal**, not as a replacement for the market state.

If `hybrid` consistently beats `market-only`, that is already a strong argument for the multimodal benchmark story.